In [ ]:
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.preprocessing import (
    StandardScaler,
    RobustScaler,
    MinMaxScaler,
    MaxAbsScaler,
    OneHotEncoder,
    LabelEncoder,
    Normalizer,
)
from sklearn.metrics import mean_absolute_error,mean_squared_error,accuracy_score
from sklearn.model_selection import cross_val_score,train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression,LogisticRegressionCV,RidgeClassifier,RidgeClassifierCV
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier,AdaBoostClassifier,VotingClassifier,BaggingClassifier,RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier,XGBRFClassifier
from catboost import CatBoostClassifier,CatBoost
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import ExtraTreesClassifier
import wandb
np.random.seed(42)

In [ ]:
PROJECT_NAME = 'Titanic-V3'

In [ ]:
def validate(model,X,y):
    preds = model.predict(X)
    results = {
        'Acc':accuracy_score(y,preds),
        'MAE':mean_absolute_error(y,preds),
        'MSE':mean_squared_error(y,preds)
    }
    return results
def fit_model(model,X_train,y_train,X_test,y_test,name):
    wandb.init(project=PROJECT_NAME,name=name)
    try:
        model = model()
    except:
        model = model
    model.fit(X_train,y_train)
    results = validate(model,X_train,y_train)
    for key,val in zip(results.keys(),results.values()):
        wandb.log({key:val})
    results = validate(model,X_test,y_test)
    for key,val in zip(results.keys(),results.values()):
        wandb.log({'Val ' + key:val})
    wandb.finish()
    return model

In [ ]:
data = pd.read_csv('./train.csv')
data = data.sample(frac=1)
test_data = pd.read_csv('./test.csv')
sample_sub = pd.read_csv('./gender_submission.csv')

In [ ]:
data.isna().sum()

In [ ]:
data['Age'].fillna(data['Age'].mean(),inplace=True)

In [ ]:
data.isna().sum()

In [ ]:
data.head()

In [ ]:
data.drop('Cabin',axis=1,inplace=True)

In [ ]:
new_tickets = []
for ticket in data['Ticket']:
    try:
        ticket = int(ticket.split(' ')[0])
    except:
        try:
            ticket = int(ticket.split(' ')[1])
        except:
            try:
                ticket = int(ticket.split(' ')[2])
            except:
                ticket = np.mean(new_tickets)
    new_tickets.append(ticket)
data['Ticket'] = new_tickets

In [ ]:
data.dtypes

In [ ]:
new_names = []
for name in data['Name']:
    new_names.append(name.split(', ')[1].split('.')[0])
data['Name'] = new_names

In [ ]:
X = data.drop('Survived',axis=1)
y = data['Survived']
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.125)

In [ ]:
from sklearn.compose import make_column_transformer

In [ ]:
len(data.drop(['Name','Sex','Embarked'],axis=1).columns)

In [ ]:
data.drop(['Pclass','Fare'],axis=1,inplace=True)

In [ ]:
ohe = make_column_transformer(
    (OneHotEncoder(handle_unknown='ignore'),['Name','Sex','Embarked']),
#     (StandardScaler(),['PassengerId', 'Age', 'SibSp', 'Parch', 'Ticket','Fare'])
    remainder='passthrough'
)
ohe.fit(X_train)
X_train = ohe.transform(X_train)

In [ ]:
X_train[1]

In [ ]:
X_test = ohe.transform(X_test)

In [ ]:
# model = fit_model(GradientBoostingClassifier,X_train,y_train,X_test,y_test,'baseline')

In [ ]:
# for col in ['Pclass']:
#     max_num = data[col].quantile(0.99)
#     min_num = data[col].quantile(0.05)
#     data = data[data[col] < max_num]
#     data = data[data[col] > min_num]
#     print(len(data))
#     X = data.drop('Survived',axis=1)
#     y = data['Survived']
#     X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.5)
#     ohe = make_column_transformer(
#         (OneHotEncoder(handle_unknown='ignore'),['Name','Sex','Embarked']),
#         remainder='passthrough'
#     )
#     ohe.fit(X_train)
#     X_train = ohe.transform(X_train)
#     X_test = ohe.transform(X_test)
# fit_model(GradientBoostingClassifier,X_train,y_train,X_test,y_test,f'baseline')

In [ ]:
pres = [StandardScaler(),RobustScaler(),MinMaxScaler(),MaxAbsScaler(),OneHotEncoder(),Normalizer()]

In [ ]:
X = data.drop('Survived',axis=1)
y = data['Survived']
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.5)
ohe = make_column_transformer(
    (OneHotEncoder(handle_unknown='ignore'),['Name','Sex','Embarked']),
    remainder='passthrough'
)
ohe.fit(X_train)
X_train = ohe.transform(X_train)
X_test = ohe.transform(X_test)

In [ ]:
models = [
    ['KNeighborsClassifier',KNeighborsClassifier],
    ['LogisticRegression',LogisticRegression],
    ['LogisticRegressionCV',LogisticRegressionCV],
    ['RidgeClassifier',RidgeClassifier],
    ['RidgeClassifierCV',RidgeClassifierCV],
    ['GaussianNB',GaussianNB],
    ['DecisionTreeClassifier',DecisionTreeClassifier],
    ['GradientBoostingClassifier',GradientBoostingClassifier],
    ['AdaBoostClassifier',AdaBoostClassifier],
    ['RandomForestClassifier',RandomForestClassifier],
    ['BaggingClassifier',BaggingClassifier],
    ['SVC',SVC],
    ['XGBClassifier',XGBClassifier],
    ['XGBRFClassifier',XGBRFClassifier],
    ['CatBoostClassifier',CatBoostClassifier],
]

In [ ]:
# for model in models:
#     fit_model(model[1],X_train,y_train,X_test,y_test,f'{model[0]}')

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
model = RandomForestClassifier(bootstrap=True,class_weight='balanced',criterion= 'entropy',max_features= 'log2',n_estimators= 500,oob_score= True,warm_start= True)
model = fit_model(model,X_train,y_train,X_test,y_test,f'Final')
import pickle
pickle.dump(model,open('./model.pkl','wb'))

In [ ]:
test_data.isna().sum()

In [ ]:
test_data.dtypes

In [ ]:
del test_data['Cabin']

In [ ]:
new_tickets = []
for ticket in test_data['Ticket']:
    try:
        ticket = int(ticket.split(' ')[0])
    except:
        try:
            ticket = int(ticket.split(' ')[1])
        except:
            try:
                ticket = int(ticket.split(' ')[2])
            except:
                ticket = np.mean(new_tickets)
    new_tickets.append(ticket)
test_data['Ticket'] = new_tickets

In [ ]:
new_names = []
for name in test_data['Name']:
    new_names.append(name.split(', ')[1].split('.')[0])
test_data['Name'] = new_names

In [ ]:
corr_matrix = data.corr()
fig,ax = plt.subplots(figsize=(15,10))
ax = sns.heatmap(corr_matrix,annot=True,linewidths=0.5,fmt='.2f',cmap='YlGnBu')

In [ ]:
test_data.drop(['Pclass','Fare'],axis=1,inplace=True)

In [ ]:
ids = pd.read_csv('./test.csv')['PassengerId']

In [ ]:
test_data['Age'] = test_data['Age'].fillna(test_data['Age'].mean())

In [ ]:
test_data.isna().sum()

In [ ]:
test_data = ohe.transform(test_data)

In [ ]:
preds = model.predict(test_data)

In [ ]:
sample_sub

In [ ]:
df = pd.DataFrame({'PassengerId':ids,'Survived':preds})

In [ ]:
df.head()

In [ ]:
df.to_csv('./submission-0.csv',index=False)

In [ ]:
from sklearn.model_selection import cross_val_score

In [ ]:
np.mean(cross_val_score(model,X_train,y_train,cv=100))

In [ ]:
np.mean(cross_val_score(model,X_test,y_test,cv=100))